# 02 Indexation et Base Vectorielle ChromaDB

## Objectif
Transformer nos 652 abstracts PubMed en vecteurs numériques (embeddings) et les stocker dans ChromaDB pour la recherche sémantique.

## Pipeline d'indexation

652 abstracts PubMed -> Chunking adaptatif(abstracts courts → 1 chunk / abstracts longs → plusieurs chunks) -> Génération des embeddings (sentence-transformers : all-MiniLM-L6-v2) -> Stockage dans ChromaDB -> Base vectorielle prête pour le RAG

## Stratégie de chunking
- Abstracts < 300 mots → 1 chunk (abstract complet)
- Abstracts ≥ 300 mots → chunking récursif (300 mots, overlap 50 mots)

## Modèle d'embedding
**all-MiniLM-L6-v2** qui est modèle sentence-transformers léger et performant,
spécialement entraîné pour la similarité sémantique.

In [1]:
import subprocess
subprocess.run(["pip", "install", "langchain-text-splitters", 
                "langchain-huggingface", "langchain-chroma",
                "langchain-core"])

CompletedProcess(args=['pip', 'install', 'langchain-text-splitters', 'langchain-huggingface', 'langchain-chroma', 'langchain-core'], returncode=0)

In [2]:
# ============================================================
# 02 INDEXATION ET BASE VECTORIELLE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import os
import time
import warnings
warnings.filterwarnings('ignore')

# LangChain — imports corrigés
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Chargement des données
df = pd.read_csv('../data/processed/pubmed_articles.csv')

print(f"Shape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")
print(f"\nAperçu :")
print(df[['pmid', 'titre', 'annee', 'abstract_words']].head())

Shape : (652, 8)
Colonnes : ['pmid', 'titre', 'abstract', 'annee', 'journal', 'requete', 'abstract_length', 'abstract_words']

Aperçu :
       pmid                                              titre annee  \
0  40160383  Exploring prevalence and factors associated wi...  2025   
1  40077974  Employment Trajectories of Recently Certified ...  2025   
2  39972449  Exploring the unemployment crisis among speech...  2025   
3  39935739  Mediating role of perceived social support in ...  2024   
4  38784586  Analyzing the impact of unemployment on mental...  2024   

   abstract_words  
0             244  
1             281  
2             380  
3             215  
4             254  


## Chunking 

Stratégie :
- Abstracts < 300 mots → 1 chunk (abstract complet)
- Abstracts ≥ 300 mots → chunking récursif (300 mots, overlap 50 mots)

On conserve les métadonnées (pmid, titre, année, journal) dans chaque chunk pour pouvoir citer les sources dans les réponses.

In [3]:
# ============================================================
# Chunking
# ============================================================

# Initialisation du text splitter pour les abstracts longs
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1500,  # ~300 mots en caractères
    chunk_overlap = 250,   # ~50 mots d'overlap
    separators    = ["\n\n", "\n", ". ", " ", ""]
)

documents = []
stats     = {"courts": 0, "longs": 0, "chunks_total": 0}

for _, row in df.iterrows():
    
    # Métadonnées à conserver dans chaque chunk
    metadata = {
        "pmid"   : str(row['pmid']),
        "titre"  : str(row['titre']),
        "annee"  : str(row['annee']),
        "journal": str(row['journal']),
        "requete": str(row['requete']),
        "source" : f"PubMed PMID:{row['pmid']}"
    }
    
    # Texte complet = titre + abstract
    texte_complet = f"{row['titre']}. {row['abstract']}"
    
    if row['abstract_words'] < 300:
        # Abstract court → 1 seul chunk
        documents.append(Document(
            page_content=texte_complet,
            metadata=metadata
        ))
        stats["courts"] += 1
        stats["chunks_total"] += 1
    else:
        # Abstract long → chunking récursif
        chunks = text_splitter.split_text(texte_complet)
        for i, chunk in enumerate(chunks):
            meta_chunk = metadata.copy()
            meta_chunk["chunk_id"] = i
            documents.append(Document(
                page_content=chunk,
                metadata=meta_chunk
            ))
        stats["longs"]        += 1
        stats["chunks_total"] += len(chunks)

print("=== RÉSULTATS DU CHUNKING ===")
print(f"Abstracts courts (< 300 mots) : {stats['courts']:,} → 1 chunk chacun")
print(f"Abstracts longs  (≥ 300 mots) : {stats['longs']:,} → plusieurs chunks")
print(f"Total chunks générés          : {stats['chunks_total']:,}")
print(f"Ratio chunks/articles         : {stats['chunks_total']/len(df):.2f}")

# Aperçu d'un document
print(f"\n=== APERÇU D'UN DOCUMENT ===")
print(f"Contenu : {documents[0].page_content[:200]}...")
print(f"Métadonnées : {documents[0].metadata}")

=== RÉSULTATS DU CHUNKING ===
Abstracts courts (< 300 mots) : 518 → 1 chunk chacun
Abstracts longs  (≥ 300 mots) : 134 → plusieurs chunks
Total chunks générés          : 836
Ratio chunks/articles         : 1.28

=== APERÇU D'UN DOCUMENT ===
Contenu : Exploring prevalence and factors associated with depression and anxiety symptoms among Bangladeshi graduates: a GIS-based cross-sectional study.. Depression and anxiety are common mental health issues...
Métadonnées : {'pmid': '40160383', 'titre': 'Exploring prevalence and factors associated with depression and anxiety symptoms among Bangladeshi graduates: a GIS-based cross-sectional study.', 'annee': '2025', 'journal': 'Global mental health (Cambridge, England)', 'requete': 'graduates_unemployment_mental_health', 'source': 'PubMed PMID:40160383'}


## Génération des Embeddings

On utilise **all-MiniLM-L6-v2** de sentence-transformers :

| Caractéristique | Valeur |
|----------------|--------|
| Dimensions | 384 |
| Taille du modèle | ~80MB |
| Vitesse | Très rapide (CPU) |
| Spécialité | Similarité sémantique |

On a choisi ce modèle car il est léger, gratuit, et fonctionne en local sans GPU 
et donne d'excellents résultats pour la recherche sémantique.

In [4]:
# ============================================================
# Initialisation du modèle d'embedding
# ============================================================

print("Chargement du modèle d'embedding...")
print("(téléchargement ~80MB au premier lancement)\n")

embedding_model = HuggingFaceEmbeddings(
    model_name      = "all-MiniLM-L6-v2",
    model_kwargs    = {"device": "cpu"},
    encode_kwargs   = {"normalize_embeddings": True}
)

# Test rapide
test_embedding = embedding_model.embed_query(
    "depression in unemployed graduates"
)

print(f"Dimensions des embeddings : {len(test_embedding)}")
print(f"Exemple (5 premières valeurs) : {test_embedding[:5]}")

Chargement du modèle d'embedding...
(téléchargement ~80MB au premier lancement)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2807.18it/s]


Dimensions des embeddings : 384
Exemple (5 premières valeurs) : [-0.006710328161716461, 0.06735158711671829, 0.028355427086353302, 0.10730835795402527, 0.05508287250995636]


## Création de la base vectorielle ChromaDB

On indexe les 836 chunks dans ChromaDB.nChaque chunk est transformé en vecteur de 384 dimensions et stocké avec ses métadonnées (pmid, titre, année, journal).

In [5]:
# ============================================================
# Création de la base vectorielle ChromaDB
# ============================================================

import chromadb

# Dossier de persistance de la base
CHROMA_DIR = "../data/processed/chroma_db"

# Suppression de l'ancienne base si elle existe
import shutil
if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)

print("Création de la base ChromaDB...")
print(f"Nombre de chunks à indexer : {len(documents):,}")

start_time = time.time()

# Création de la base vectorielle
vectorstore = Chroma.from_documents(
    documents        = documents,
    embedding        = embedding_model,
    persist_directory= CHROMA_DIR,
    collection_name  = "pubmed_biomedical"
)

elapsed = time.time() - start_time

print(f"Temps d'indexation : {elapsed:.1f} secondes")
print(f"Chunks indexés     : {vectorstore._collection.count():,}")
print(f"Dossier            : {CHROMA_DIR}")

Création de la base ChromaDB...
Nombre de chunks à indexer : 836
Temps d'indexation : 74.5 secondes
Chunks indexés     : 836
Dossier            : ../data/processed/chroma_db


## Test de la recherche sémantique

On va testé la base vectorielle avec des questions réelles pour vérifier que les chunks récupérés sont pertinents.

In [6]:
# ============================================================
# Test de la recherche sémantique
# ============================================================

questions_test = [
    "What is the impact of unemployment on mental health of graduates?",
    "Does job insecurity cause depression in young adults?",
    "What are the psychological effects of post-graduation transition?",
    "How does precarious employment affect wellbeing of educated workers?"
]

print("=== TEST DE RECHERCHE SÉMANTIQUE ===\n")

for question in questions_test:
    print(f"❓ Question : {question}")
    
    # Recherche des 3 chunks les plus pertinents
    resultats = vectorstore.similarity_search_with_score(
        query = question,
        k     = 3
    )
    
    print(f"\nTop 3 résultats :")
    for i, (doc, score) in enumerate(resultats):
        print(f"\n  [{i+1}] Score similarité : {score:.4f}")
        print(f"       PMID    : {doc.metadata['pmid']}")
        print(f"       Année   : {doc.metadata['annee']}")
        print(f"       Journal : {doc.metadata['journal'][:50]}")
        print(f"       Titre   : {doc.metadata['titre'][:80]}...")
        print(f"       Extrait : {doc.page_content[:150]}...")
    
    print(f"\n{'─'*60}\n")
    time.sleep(0.5)

=== TEST DE RECHERCHE SÉMANTIQUE ===

❓ Question : What is the impact of unemployment on mental health of graduates?

Top 3 résultats :

  [1] Score similarité : 0.5655
       PMID    : 33359536
       Année   : 2021
       Journal : Annals of epidemiology
       Titre   : Effects of graduating during economic downturns on mental health....
       Extrait : Effects of graduating during economic downturns on mental health.. This study examined the effects of economic downturns at the time of graduation on ...

  [2] Score similarité : 0.6166
       PMID    : 37957781
       Année   : 2024
       Journal : The International journal of health planning and m
       Titre   : Young graduates and economic recession: Lessons from the pandemic to prevent the...
       Extrait : Young graduates and economic recession: Lessons from the pandemic to prevent the (re)incidence of mental health symptoms.. Economic conditions affect ...

  [3] Score similarité : 0.6917
       PMID    : 26792092
      

In [7]:
# ============================================================
# Résumé final de l'indexation
# ============================================================

print("=== RÉSUMÉ FINAL DE L'INDEXATION ===")
print(f"Articles PubMed collectés : 652")
print(f"Chunks générés            : 836")
print(f"Dimensions embeddings     : 384")
print(f"Modèle embedding          : all-MiniLM-L6-v2")
print(f"Base vectorielle          : ChromaDB")
print(f"Localisation              : data/processed/chroma_db/")

=== RÉSUMÉ FINAL DE L'INDEXATION ===
Articles PubMed collectés : 652
Chunks générés            : 836
Dimensions embeddings     : 384
Modèle embedding          : all-MiniLM-L6-v2
Base vectorielle          : ChromaDB
Localisation              : data/processed/chroma_db/
